# VectorForge Part 2 — OCR GPU Benchmark
This notebook drives the reusable CRNN + CTC library to study CPU/GPU hardware, batch size, precision, and image resolution. The OCR model is the workload; the goal is hardware-aware measurement.

**Kaggle:** open Notebook Settings → Accelerator → GPU. **Colab:** Runtime → Change runtime type → GPU.

## 2–3. Environment Setup and Verify NVIDIA GPU

In [ ]:
import subprocess, sys
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi is unavailable; PyTorch CUDA detection remains authoritative.')

## 4. Clone / Import VectorForge
Set `REPO_URL` only when the repository is not already attached to the notebook.

In [ ]:
from pathlib import Path
REPO_URL = ''  # e.g. https://github.com/OWNER/VectorForge.git
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'config' / 'ocr.yaml').exists():
    if not REPO_URL:
        raise RuntimeError('Attach/clone VectorForge or set REPO_URL.')
    subprocess.run(['git', 'clone', REPO_URL, 'VectorForge'], check=True)
    PROJECT_DIR = Path('VectorForge').resolve()
sys.path.insert(0, str(PROJECT_DIR))
%cd $PROJECT_DIR

## 5. Install Dependencies
The cell deliberately preserves the cloud runtime's CUDA-enabled PyTorch build.

In [ ]:
import importlib.util
required = {'yaml':'PyYAML', 'pandas':'pandas', 'PIL':'Pillow', 'matplotlib':'matplotlib', 'tqdm':'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
print('Missing non-PyTorch packages:', missing)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

## 6. Configuration
Smoke mode is intentionally tiny. Change the full-mode values in this one cell before spending GPU quota.

In [ ]:
import copy, yaml
base_config = yaml.safe_load(Path('config/ocr.yaml').read_text())
smoke_config = copy.deepcopy(base_config)
smoke_config['paths'].update(generated_dir='data/ocr/generated/smoke', metadata_file='data/ocr/metadata/smoke.csv', results_dir='results/ocr/smoke')
smoke_config['dataset'].update(train_samples=500, validation_samples=100, test_samples=100, max_text_length=24)
smoke_config['image'].update(width=128, height=32, font_size=16, augment=False)
smoke_config['model'].update(cnn_channels=[16, 32], hidden_size=32, lstm_layers=1)
smoke_config['training'].update(epochs=1, batch_size=16, num_workers=0)
full_config = copy.deepcopy(base_config)
full_config['dataset'].update(train_samples=20000, validation_samples=2000, test_samples=2000)
full_config['training'].update(epochs=3, batch_size=32, num_workers=2)
full_config['benchmark'].update(batch_sizes=[8,16,32,64,128], precisions=['fp32','fp16','bf16'], resolutions=[[128,32],[256,64],[512,128]])

## 7. Generate or Load Dataset

In [ ]:
from src.ocr.data.synthetic_generator import GenerationConfig, generate_dataset, load_source_sentences
from src.ocr.data.vocabulary import CharacterVocabulary
vocabulary = CharacterVocabulary()
cfg = smoke_config
sentences = load_source_sentences(cfg['paths'].get('source_text'), cfg['dataset']['text_column'], vocabulary, cfg['dataset']['max_text_length'])
generation = GenerationConfig(output_dir=Path(cfg['paths']['generated_dir']), metadata_file=Path(cfg['paths']['metadata_file']), width=cfg['image']['width'], height=cfg['image']['height'], font_size=cfg['image']['font_size'], padding=cfg['image']['padding'], train_samples=cfg['dataset']['train_samples'], validation_samples=cfg['dataset']['validation_samples'], test_samples=cfg['dataset']['test_samples'], max_text_length=cfg['dataset']['max_text_length'], seed=cfg['seed'])
metadata = generate_dataset(generation, sentences, vocabulary)
metadata.groupby('split').size()

## 8. Dataset Visualization

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 3, figsize=(14, 4))
for ax, (_, row) in zip(axes.flat, metadata.head(6).iterrows()):
    ax.imshow(Image.open(row.image_path), cmap='gray')
    ax.set_title(row.text); ax.axis('off')
plt.tight_layout()

## 9. Build OCR Model
CNN features become a left-to-right sequence, the BiLSTM adds context, and CTC learns alignment without character bounding boxes.

In [ ]:
from src.ocr.models.model_factory import create_ocr_model
model = create_ocr_model(smoke_config['model'], vocabulary.size)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 10. CPU Smoke Test

In [ ]:
from src.ocr.training.trainer import train_from_config
cpu_config = copy.deepcopy(smoke_config); cpu_config['training'].update(device='cpu', precision='fp32')
cpu_result, cpu_history, last_model = train_from_config(cpu_config, experiment_id='notebook-cpu-smoke')
cpu_result.to_dict()

## 11–12. GPU Smoke Test and Baseline GPU Training
CUDA timing is synchronized by the trainer; reported throughput includes loading, host-to-device transfer, forward, CTC, backward, and optimizer work.

In [ ]:
gpu_result = None
if torch.cuda.is_available():
    gpu_config = copy.deepcopy(smoke_config); gpu_config['training'].update(device='cuda', precision='fp32')
    gpu_result, gpu_history, last_model = train_from_config(gpu_config, experiment_id='notebook-gpu-smoke')
    print(gpu_result.to_dict())
else:
    print('Enable a GPU accelerator before running GPU experiments.')

## 13–16. CPU vs GPU, Batch Size, Precision, and Resolution Benchmarks
Larger batches expose more parallel GPU work but consume more VRAM. FP16/BF16 can reduce storage and use Tensor Cores on supported NVIDIA hardware. Keep `RUN_FULL_BENCHMARKS=False` until smoke tests pass.

In [ ]:
from src.ocr.benchmarks.ocr_benchmark import run_benchmark_matrix
RUN_FULL_BENCHMARKS = False
if RUN_FULL_BENCHMARKS:
    full_sentences = load_source_sentences(full_config['paths'].get('source_text'), full_config['dataset']['text_column'], vocabulary, full_config['dataset']['max_text_length'])
    full_generation = GenerationConfig(output_dir=Path(full_config['paths']['generated_dir']), metadata_file=Path(full_config['paths']['metadata_file']), width=full_config['image']['width'], height=full_config['image']['height'], font_size=full_config['image']['font_size'], padding=full_config['image']['padding'], train_samples=full_config['dataset']['train_samples'], validation_samples=full_config['dataset']['validation_samples'], test_samples=full_config['dataset']['test_samples'], max_text_length=full_config['dataset']['max_text_length'], seed=full_config['seed'])
    generate_dataset(full_generation, full_sentences, vocabulary)
    benchmark_rows = run_benchmark_matrix(full_config, {'baseline','batch_size','precision','resolution'})
else:
    print('Full benchmark disabled; set the flag after validating the smoke run.')

## 17. Evaluate CER / WER and Inspect Predictions

In [ ]:
from src.ocr.training.trainer import create_data_loaders
from src.ocr.training.losses import CTCLossWithValidation
from src.ocr.evaluation.evaluator import evaluate_model
from src.ocr.training.precision import resolve_device
_, val_loader = create_data_loaders(smoke_config['paths']['metadata_file'], vocabulary, width=128, height=32, augment=False, batch_size=16, num_workers=0, pin_memory=False, seed=42)
eval_device = next(last_model.parameters()).device
evaluation = evaluate_model(last_model, val_loader, CTCLossWithValidation(), vocabulary, eval_device)
print({k:v for k,v in evaluation.items() if k not in {'references','predictions'}})
for truth, prediction in list(zip(evaluation['references'], evaluation['predictions']))[:5]:
    print(f'Ground truth: {truth!r}\nPrediction:   {prediction!r}\n')

## 18. Generate Plots

In [ ]:
from src.visualization.ocr_plots import generate_ocr_plots
results_file = Path('results/ocr/benchmark_results.csv')
if results_file.exists():
    plot_files = generate_ocr_plots(results_file, 'results/ocr/plots')
    print(*plot_files, sep='\n')
else:
    print('Run the benchmark matrix to create comparison plots.')

## 19. Export Results
Artifacts include benchmark/training CSVs, checkpoints, metadata, and plots. Kaggle exposes notebook output files after a saved run.

In [ ]:
import shutil
archive = shutil.make_archive('vectorforge-ocr-results', 'zip', 'results/ocr')
print('Export archive:', archive)
try:
    from google.colab import files
    # Uncomment in Colab: files.download(archive)
except ImportError:
    pass

## 20. Experiment Summary
Compare throughput, peak allocated VRAM, CER, and WER—not speed alone. Record the actual assigned GPU, because hosted notebook hardware varies. Perfect numerical identity across CPU, CUDA, FP16, and BF16 is not expected. Small workloads may underutilize a GPU because launch and transfer overhead dominate.